# Create Campaign Normalization Values

This notebook runs the unoptimized baseline simulations defined in `campaign_config.yaml`, compares their summary statistics, and writes campaign-wide normalization scalars to `campaign_normalization.yaml`.
The resulting normalization values can be passed into each optimization run so objective weights stay comparable across operating modes and strategies.

## 1 - Setup and Imports

In [2]:
import yaml
import pandas as pd
from copy import deepcopy
from pathlib import Path

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.perfusion_strategies import PerfusionStrategy
from animakinetica.models.gas_controllers import GasController

CONFIG_FILE_NAME = "campaign_config.yaml"
NORM_FILE_NAME = "campaign_normalization.yaml"


def merge_config_layer(base, override):
    """Merge one config layer into another, appending parameter_sets."""
    result = deepcopy(base)

    for key, value in override.items():
        if key == "parameter_sets":
            existing = result.get(key, []) or []
            additions = deepcopy(value) if value else []
            result[key] = [*existing, *additions]
        elif isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = merge_config_layer(result[key], value)
        else:
            result[key] = deepcopy(value)

    return result


def resolve_simulation_config(campaign_config, sim_entry):
    """Merge campaign defaults, mode defaults, and simulation overrides."""
    merged_config = {}
    defaults = campaign_config.get("simulation_defaults", {})
    mode_defaults = campaign_config.get("mode_defaults", {}).get(sim_entry.get("mode"), {})

    for layer in (defaults, mode_defaults, sim_entry):
        merged_config = merge_config_layer(merged_config, layer)

    return merged_config


# Resolve campaign directory
_cwd = Path().resolve()
if (_cwd / CONFIG_FILE_NAME).exists():
    CAMPAIGN_DIR = _cwd
elif (_cwd.parent / CONFIG_FILE_NAME).exists():
    CAMPAIGN_DIR = _cwd.parent
else:
    raise FileNotFoundError(
        f"Cannot find {CONFIG_FILE_NAME}. Please open this notebook from the Campaign 4 directory."
    )

PARAMS_DIR = CAMPAIGN_DIR / "parameters"

print(f"Campaign dir : {CAMPAIGN_DIR}")
print(f"Parameters   : {PARAMS_DIR}")
print(f"Config file  : {CONFIG_FILE_NAME}")

Campaign dir : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_1_fixed_stoichiometry_Humbird
Parameters   : C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_1_fixed_stoichiometry_Humbird\parameters
Config file  : campaign_config.yaml


## 2 - Load Campaign Configuration

In [3]:
with open(CAMPAIGN_DIR / CONFIG_FILE_NAME, encoding="utf-8") as f:
    campaign_config = yaml.safe_load(f)

simulations = [
    resolve_simulation_config(campaign_config, sim_entry)
    for sim_entry in campaign_config["simulations"]
]

print(f"Loaded {len(simulations)} simulations from campaign config:\n")
for s in simulations:
    strategies = " | ".join(filter(None, [s.get("feed_strategy"), s.get("perfusion_strategy")])) or "-"
    print(f"  [{s['id']:24s}]  {s['mode']:25s}  {strategies}")

Loaded 17 simulations from campaign config:

  [fedbatch_constant       ]  FEDBATCH                   CONSTANT
  [fedbatch_linear         ]  FEDBATCH                   LINEAR
  [fedbatch_exponential    ]  FEDBATCH                   EXPONENTIAL
  [fedbatch_csfr           ]  FEDBATCH                   CSFR
  [fedbatch_gsp            ]  FEDBATCH                   GLUCOSE_SETPOINT
  [batch                   ]  BATCH_WITH_STARTUP_FEDBATCH  CONSTANT
  [chemostat               ]  CHEMOSTAT                  CONSTANT
  [cf_constant             ]  CONCENTRATED_FEDBATCH      CONSTANT | CONSTANT
  [cf_tbs                  ]  CONCENTRATED_FEDBATCH      CONSTANT | TOXICITY_BASED_STEP
  [cf_tbp                  ]  CONCENTRATED_FEDBATCH      CONSTANT | TOXICITY_BASED_PROPORTIONAL
  [cf_cspr                 ]  CONCENTRATED_FEDBATCH      CONSTANT | CSPR
  [cf_gsp                  ]  CONCENTRATED_FEDBATCH      CONSTANT | GLUCOSE_SETPOINT
  [cp_constant             ]  CONTINUOUS_PERFUSION       CONSTANT |

## 3 - Run All Baseline Simulations

Each simulation is resolved from the layered campaign configuration and run once without optimization. Results are stored in `results`, keyed by simulation ID.

In [4]:
results = {}
errors = {}

for sim_cfg in simulations:
    sim_id = sim_cfg["id"]
    label = sim_cfg["label"]
    print(f"\n{'-' * 72}")
    print(f"  {sim_id}")
    print(f"  {label}")
    print(f"{'-' * 72}")

    try:
        mode = OperatingMode[sim_cfg["mode"]]
        kinetics = Kinetics[sim_cfg["kinetics"]]
        feed_strategy = FeedStrategy[sim_cfg["feed_strategy"]] if sim_cfg.get("feed_strategy") else None
        perfusion_strategy = (
            PerfusionStrategy[sim_cfg["perfusion_strategy"]]
            if sim_cfg.get("perfusion_strategy")
            else None
        )
        solver_method = sim_cfg.get("solver_method", "LSODA")

        param_files = [
            PARAMS_DIR / Path(entry).with_suffix(".yaml")
            for entry in sim_cfg["parameter_sets"]
        ]
        params = SimulationParameters.load_parameters([str(path) for path in param_files])

        sim = CellCultureSimulation(
            parameters=params,
            kinetics=kinetics,
            operating_mode=mode,
            feed_strategy=feed_strategy,
            perfusion_strategy=perfusion_strategy,
            solver_method=solver_method,
            gas_controller=GasController.CASCADE_N_FG_XO2,
        )
        sim.run_simulation()

        results[sim_id] = sim.data
        stats = sim.data.summary_stats
        print(
            f"  OK  STY = {stats.get('STY_kg_m3_day', float('nan')):.4f} kg/m^3/day  "
            f"| Xv_final ~= {sim.data.batches[-1].Xv_final:.2e} cells/L  "
            f"| t_end = {sim.data.batches[-1].t_end:.1f} h"
        )

    except Exception as exc:
        errors[sim_id] = exc
        print(f"  FAILED: {exc}")

print(f"\n{'=' * 72}")
print(f"  Completed: {len(results)} / {len(simulations)} simulations")
if errors:
    print(f"  Failed   : {list(errors.keys())}")


------------------------------------------------------------------------
  fedbatch_constant
  Fed-Batch | Humbird Wildtype | Constant Feed
------------------------------------------------------------------------
Loading parameters from 5 file(s) sequentially...
  [1/5] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_1_fixed_stoichiometry_Humbird\parameters\base_parameters.yaml
  [2/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_1_fixed_stoichiometry_Humbird\parameters\stoichiometry\anab_Humbird_cell.yaml
  [3/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_1_fixed_stoichiometry_Humbird\parameters\stoichiometry\cata_Humbird_wildtype_yield_based.yaml
  [4/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_1_fixed_stoichiometry_Humbird\parameters\cell_properties\cell_H

## 4 — Compare Summary Statistics

In [ ]:
# Build tidy comparison DataFrame
METRIC_LABELS = {
    "STY_kg_m3_day": "STY  [kg/m^3/day]",
    "medium_conversion_efficiency_m3_kg": "Medium conversion  [m^3/kg DCM]",
    "feed_conversion_efficiency_kg_kg": "Feed conversion  [kg feed/kg DCM]",
    "protein_conversion_efficiency_kg_kg": "Protein conversion  [kg Gln+AA/kg protein]",
    "total_DCM_harvested_kg": "Total Biomass  [kg]",
    "total_simulation_time_h": "Total Simulation Time  [h]",
    "number_of_batches": "Number of Batches",
}

# Metrics that live in simulated_window_totals_stats rather than summary_stats
_TOTALS_ONLY_KEYS = {"total_DCM_harvested_kg", "total_simulation_time_h", "number_of_batches"}

rows = []
for sim_cfg in simulations:
    sid = sim_cfg["id"]
    if sid not in results:
        continue
    stats = results[sid].summary_stats
    totals = results[sid].simulated_window_totals_stats
    row = {
        "ID": sid,
        "Mode": sim_cfg["mode"],
        "Feed": sim_cfg.get("feed_strategy") or "-",
        "Perf.": sim_cfg.get("perfusion_strategy") or "-",
    }
    for key in METRIC_LABELS:
        if key in _TOTALS_ONLY_KEYS:
            row[key] = totals.get(key, float("nan"))
        else:
            row[key] = stats.get(key, float("nan"))
    rows.append(row)

df = pd.DataFrame(rows).set_index("ID")
df_display = df.rename(columns=METRIC_LABELS)

float_fmts = {
    "STY  [kg/m^3/day]": "{:.4f}",
    "Medium conversion  [m^3/kg DCM]": "{:.3f}",
    "Feed conversion  [kg feed/kg DCM]": "{:.4f}",
    "Protein conversion  [kg Gln+AA/kg protein]": "{:.4f}",
    "Total Biomass  [kg]": "{:.4f}",
    "Total Simulation Time  [h]": "{:.1f}",
    "Number of Batches": "{:.0f}",
}

In [ ]:
# Ranked view: sort by STY descending; lower conversion costs are favourable.
sty_col = "STY  [kg/m^3/day]"
conversion_cols = [
    "Medium conversion  [m^3/kg DCM]",
    "Feed conversion  [kg feed/kg DCM]",
    "Protein conversion  [kg Gln+AA/kg protein]",
]

df_ranked = (
    df_display
    .sort_values(sty_col, ascending=False)
    .assign(Rank=range(1, len(df_display) + 1))
    .set_index("Rank")
)

(
    df_ranked.style
    .background_gradient(subset=[sty_col], cmap="YlGn")
    .background_gradient(subset=conversion_cols, cmap="YlGn_r")
    .format(float_fmts, na_rep="-")
    .set_caption("Simulations Ranked by STY (unoptimized; lower conversion costs are better)")
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "14px"), ("font-weight", "bold"), ("text-align", "left"), ("padding-bottom", "8px")]},
        {"selector": "th",
         "props": [("background-color", "#2c5f8a"), ("color", "white"), ("text-align", "center"), ("white-space", "nowrap")]},
        {"selector": "td",
         "props": [("text-align", "center"), ("padding", "5px 10px")]},
    ])
)

,Mode,Feed,Perf.,STY [kg/m^3/day],Medium efficiency [kg/m^3],Nutrient efficiency [kg/kg],Protein efficiency [kg/kg],Total Biomass [kg],Total Simulation Time [h],Number of Batches
Rank,,,,,,,,,,
1,CONTINUOUS_PERFUSION,CONSTANT,TOXICITY_BASED_PROPORTIONAL,2.7574,0.729,0.1302,0.2570,220.5941,960.0,1
2,CONTINUOUS_PERFUSION,CONSTANT,TOXICITY_BASED_STEP,2.7569,0.725,0.1296,0.2558,220.5558,960.0,1
3,CONTINUOUS_PERFUSION,CONSTANT,GLUCOSE_SETPOINT,2.7301,0.767,0.1370,0.2703,218.4119,960.0,1
4,CONCENTRATED_FEDBATCH,CONSTANT,TOXICITY_BASED_PROPORTIONAL,2.2127,0.800,0.1416,0.2768,177.0198,960.0,2
5,CONCENTRATED_FEDBATCH,CONSTANT,TOXICITY_BASED_STEP,2.2114,0.790,0.1399,0.2735,176.9155,960.0,2
6,CONCENTRATED_FEDBATCH,CONSTANT,GLUCOSE_SETPOINT,2.1316,0.910,0.1607,0.3132,170.5312,960.0,2
7,CONTINUOUS_PERFUSION,CONSTANT,CSPR,1.9249,0.953,0.1696,0.3330,153.9914,960.0,1
8,CONCENTRATED_FEDBATCH,CONSTANT,CSPR,1.1799,1.077,0.1866,0.3567,94.3929,960.0,2
9,CONTINUOUS_PERFUSION,CONSTANT,CONSTANT,0.9025,1.157,0.2026,0.3914,72.2010,960.0,1


In [7]:
# ── Print raw summary_stats keys for any sim (useful for debugging) ───────────
if results:
    first_id = next(iter(results))
    print(f"All summary_stats keys available (from '{first_id}'):")
    for k, v in results[first_id].summary_stats.items():
        print(f"  {k:45s}: {v}")

All summary_stats keys available (from 'fedbatch_constant'):
  total_DCM_harvested_kg                       : 99.73376293286104
  STY_kg_m3_day                                : 0.1558340045825954
  medium_use_efficiency_kg_m3                  : 1.5853418302649387
  protein_use_efficiency_kg_kg                 : 0.20409854662649143
  nutrient_use_efficiency_kg_kg                : 0.15331654559780217
  batch_time_range_h                           : 103.28257691756752


## 5 - Compute and Save Campaign Normalization Scalars

Computes the maximum value for each optimization objective across all successful baseline simulations and saves the result to `campaign4_normalization.yaml` in the campaign root.

Each simulation script loads this file and passes it to `optimize_parameters(normalization_values=...)`, so every run divides its objectives by the same campaign-wide constants.

In [ ]:
# Objectives must match campaign_config.yaml. Divisors use the largest finite
# magnitude in the campaign; objective direction is handled by OptimizationManager.
NORM_OBJECTIVES = campaign_config["optimization_settings"]["profiles"]["sweep_1_baseline_normalized"]["objectives"]

campaign_norm = {}
missing_in_any = set()

for obj in NORM_OBJECTIVES:
    values = []
    for sid, data in results.items():
        value = data.summary_stats.get(obj)
        if value is not None and value > 0:
            values.append(float(value))
        else:
            missing_in_any.add(f"{sid}/{obj}")
    if not values:
        raise ValueError(f"No valid baseline values found for objective '{obj}'.")
    campaign_norm[obj] = max(values)

norm_rows = []
for obj in NORM_OBJECTIVES:
    individual = {sid: results[sid].summary_stats.get(obj, float("nan")) for sid in results}
    norm_rows.append({"Objective": obj, "Campaign magnitude divisor": campaign_norm[obj], **{sid: round(value, 4) for sid, value in individual.items()}})

norm_df = pd.DataFrame(norm_rows).set_index("Objective")
print("Campaign-level normalization divisors (largest baseline magnitude):\n")
for obj, value in campaign_norm.items():
    print(f"  {obj:45s}: {value:.4g}")
if missing_in_any:
    print(f"\n  Warning: invalid (simulation, objective) pairs: {sorted(missing_in_any)}")

norm_df[["Campaign magnitude divisor"]].style.format("{:.4g}").set_caption("Campaign normalization divisors")

Campaign-level normalization scalars (max across all baseline simulations):

  STY_kg_m3_day                                : 2.757
  medium_use_efficiency_kg_m3                  : 1.585
  nutrient_use_efficiency_kg_kg                : 0.2125
  protein_use_efficiency_kg_kg                 : 0.3954


,Campaign max (divisor)
Objective,
STY_kg_m3_day,2.757
medium_use_efficiency_kg_m3,1.585
nutrient_use_efficiency_kg_kg,0.2125
protein_use_efficiency_kg_kg,0.3954


In [10]:
# Save to YAML
import yaml as _yaml
from datetime import datetime as _dt

NORM_OUT = CAMPAIGN_DIR / NORM_FILE_NAME

payload = {
    "_metadata": {
        "description": (
            "Campaign-level normalization scalars for multi-objective optimization. "
            "Each value is the maximum of the corresponding objective observed across "
            "all unoptimized baseline simulations in this campaign. "
            "Pass this mapping as normalization_values= to optimize_parameters() so "
            "weights remain comparable across operating modes."
        ),
        "generated": _dt.now().strftime("%Y-%m-%d %H:%M:%S"),
        "n_simulations_used": len(results),
        "simulation_ids": list(results.keys()),
    },
    **campaign_norm,
}

with open(NORM_OUT, "w", encoding="utf-8") as f:
    _yaml.dump(payload, f, default_flow_style=False, sort_keys=False)

print(f"Saved campaign normalization to: {NORM_OUT}")
print(f"  ({len(campaign_norm)} objectives, derived from {len(results)} baseline simulations)")

Saved campaign normalization to: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_1_fixed_stoichiometry_Humbird\campaign_normalization.yaml
  (4 objectives, derived from 17 baseline simulations)
